In [2]:
import pufferlib
from pufferlib.ocean.breakout.breakout import Breakout
import pufferlib.vector
import numpy as np
import torch
import torch.nn as nn 
from stable_baselines3.common.buffers import ReplayBuffer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import time
print(f"Using device: {device}")
from torch.distributions import Categorical
import torch.nn.functional as F
import torch
import gc

# Run garbage collector to free Python memory references
gc.collect()

# Empty the PyTorch CUDA cache
torch.cuda.empty_cache()

# Also reset the CUDA memory allocator — clears *reserved* memory
torch.cuda.ipc_collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.reset_accumulated_memory_stats()

Using device: cuda


In [3]:
class AgentNetwork(nn.Module):
    def __init__(self, vecenv, enc_hidden_size, lstm_hidden_size, lstm_num_layers, device):
        super().__init__()
        self.obs_dim = vecenv.single_observation_space.shape[0]
        self.num_action = vecenv.single_action_space.n
        self.enc_hidden_size = enc_hidden_size
        self.lstm_hidden_size = lstm_hidden_size
        self.lstm_num_layers = lstm_num_layers
        self.num_envs = vecenv.num_agents  
        self.device = device
        
        # Define layers
        self.encoder = nn.Linear(self.obs_dim, self.enc_hidden_size).to(self.device)
        self.lstm = nn.LSTM(self.enc_hidden_size, self.lstm_hidden_size, self.lstm_num_layers).to(self.device)
        self.decoder = nn.Linear(self.lstm_hidden_size, self.num_action).to(self.device)
        self.value_head = nn.Linear(self.lstm_hidden_size, 1).to(self.device)

    def init_state(self):
        h0 = torch.zeros(self.lstm_num_layers, self.num_envs, self.lstm_hidden_size, device=self.device)
        c0 = torch.zeros(self.lstm_num_layers, self.num_envs, self.lstm_hidden_size, device=self.device)
        state = (h0, c0)
        return state
    
    def forward(self, obs, state=None, Train=True): 
        if len(obs.shape) == 2:
            obs = obs.unsqueeze(0)  # Set context length = 1 when just seeing a single batch of observations
        # obs : [context_length, num_envs, obs_dim]
        context_length = obs.shape[0]

        # Initialize state if needed
       

        if Train:
            emb = torch.relu(self.encoder(obs.view(-1, self.obs_dim)))  # [context_length * num_envs, enc_hidden_size]
            lstm_outs, state = self.lstm(emb.view(context_length, self.num_envs, self.enc_hidden_size), state)
            logits = self.decoder(lstm_outs)  # [context_length, num_envs, num_actions]
            values = self.value_head(lstm_outs).squeeze(-1)  # [context_length, num_envs]
            return logits, values, state

        else:
            with torch.no_grad():
                emb = torch.relu(self.encoder(obs.view(-1, self.obs_dim)))  # [context_length * num_envs, enc_hidden_size]
                lstm_outs, state = self.lstm(emb.view(context_length, self.num_envs, self.enc_hidden_size), state)
                logits = self.decoder(lstm_outs)  # [context_length, num_envs, num_actions]
                values = self.value_head(lstm_outs).squeeze(-1)

                if context_length == 1:
                    logits = logits.squeeze(0)  # [num_envs, num_actions]
                    values = values.squeeze(0)
            return logits,values, state


In [4]:
def compute_gae(rewards, dones, values, value_Tp1, gamma, lam, trajectory_length):
    
    # Initialize advantage tensor
    A_curr = torch.zeros_like(values[0])  # shape: [num_envs]
    advantages = torch.zeros_like(values)  # shape: [trajectory_length, num_envs]

    for t in reversed(range(trajectory_length)):
        if t == trajectory_length - 1:  # last timestep
            delta = rewards[t] + (gamma * value_Tp1) - values[t]
            A_curr = delta  # No future advantage to consider, just the delta
            advantages[t] = A_curr
        else:  # middle timesteps
            delta = rewards[t] + (gamma * (1 - dones[t]) * values[t + 1]) - values[t]
            A_curr = delta + (gamma * lam * advantages[t + 1])
            advantages[t] = A_curr
    
    return advantages
    
def actor_loss(old_log_probs, new_log_probs, advantages, epsilon):
    # Calculate log ratio
    log_ratio = new_log_probs - old_log_probs
    ratio = torch.exp(log_ratio)
    
    # Compute the objective and clipped objective
    objective = ratio * advantages
    clipped_objective = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
    
    # The final actor loss is the min of the objective and clipped objective
    loss = -torch.min(objective, clipped_objective).mean()  # Mean over the batch
    return loss

def critic_loss (values, returns):
    #critic loss is just mean squared error between predicited values by the critic and return estimates. 
    return F.mse_loss(values,returns)

def total_loss(old_log_probs, new_log_probs, advantages, values, returns, entropies, epsilon, critic_coeff, entropy_coeff): 
    actorloss = actor_loss(old_log_probs, new_log_probs, advantages, epsilon)
    criticloss = critic_loss(values, returns)
    entropy_bonus = entropy_coeff * entropies.mean()  # <- Ensure it's a scalar

    totalloss = actorloss + critic_coeff * criticloss - entropy_bonus
    return totalloss

In [ ]:
# Hyperparameters

context_length = 1024
num_envs = 1024
obs_dim = 118
gamma = 0.99
lam = 0.95
epsilon = 0.2  # Clipping epsilon for PPO
critic_coeff = 0.5  # Weight for the critic loss
entropy_coeff = 0.01
learning_rate = 5e-4
n_epochs = 1000
num_train_per_epoch = 20

# Environment settings
env_kwargs = {'num_envs': num_envs}
train_kwargs = {'num_envs': 1, 'num_workers': 1, 'env_batch_size': 1}

# Agent network settings
enc_hidden_size = 64
lstm_hidden_size = 64
lstm_num_layers = 1

# Create environment
make_env = Breakout  # Assuming Breakout is defined elsewhere
vecenv = pufferlib.vector.make(
    make_env,
    env_kwargs=env_kwargs,
    num_envs=train_kwargs['num_envs'],
    num_workers=train_kwargs['num_workers'],
    batch_size=train_kwargs['env_batch_size'],
    backend=pufferlib.vector.Multiprocessing,
)

# Initialize agent
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
agent = AgentNetwork(vecenv, enc_hidden_size, lstm_hidden_size, lstm_num_layers, device)

# Initialize optimizer
optimizer = torch.optim.Adam(agent.parameters(), lr=learning_rate)


Process Process-2:
Traceback (most recent call last):
  File "/venv/main/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/venv/main/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/root/backtoRL/.venv/lib/python3.12/site-packages/pufferlib/vector.py", line 206, in _worker_process
    time.sleep(0.01)
KeyboardInterrupt


In [10]:
obs_buffer = torch.zeros(context_length, num_envs, obs_dim, device=device)
old_log_probs_buffer = torch.zeros(context_length, num_envs, device=device)
new_log_probs_buffer = torch.zeros_like(old_log_probs_buffer, device=device)
actions_buffer = torch.zeros_like(old_log_probs_buffer, device=device)
rewards_buffer = torch.zeros(context_length, num_envs, device=device)  # Shape: [context_length, num_envs]
dones_buffer = torch.zeros_like(old_log_probs_buffer, device=device)
entropies_buffer = torch.zeros_like(old_log_probs_buffer, device=device)
values_buffer = torch.zeros_like(old_log_probs_buffer, device=device)

# Reset environment and initialize state
ob, _ = vecenv.reset()
init_state = state = agent.init_state()

# Training loop
for epoch in range(n_epochs):
     

    # Collect experiences
    for step in range(context_length):
        obs_tensor = torch.tensor(ob, dtype=torch.float32, device=device)
        obs_buffer[step] = obs_tensor

        # Forward pass (no gradient calculation during inference)
        logits, _, state = agent.forward(obs_tensor, state, Train=False)

        dist = Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)

        # Step environment with chosen action
        ob, reward, done, _, _ = vecenv.step(action.cpu().numpy())

        # Store results
        rewards_buffer[step] = torch.tensor(reward, dtype=torch.float32, device=device)
        dones_buffer[step] = torch.tensor(done, dtype=torch.float32, device=device)
        actions_buffer[step] = action
        old_log_probs_buffer[step] = log_prob

        

    # Compute value for last timestep (to calculate GAE)
    _, final_value, _ = agent(torch.tensor(ob, dtype=torch.float32, device=device), state, Train=False)
    done_T = dones_buffer[-1]
    value_Tp1 = final_value * (1.0 - done_T)

    # Perform training over several epochs
    for _ in range(num_train_per_epoch):
        new_logits, values_buffer, state = agent(obs_buffer, init_state, Train=True)

        # Calculate new log probabilities and entropies for the updated policy
        new_dist = Categorical(logits=new_logits)
        new_log_probs_buffer = new_dist.log_prob(actions_buffer)
        entropies_buffer = new_dist.entropy()

        # Compute GAE
        advantages_buffer = compute_gae(rewards_buffer, dones_buffer, values_buffer, value_Tp1, gamma, lam, context_length)
        returns_buffer = advantages_buffer + values_buffer

        # Compute loss
        loss = total_loss(
            old_log_probs_buffer, new_log_probs_buffer, advantages_buffer,
            values_buffer, returns_buffer, entropies_buffer, epsilon, critic_coeff, entropy_coeff
        )

        loss.backward()
        optimizer.step()

    # Print reward information
    avg_reward = rewards_buffer.sum(dim = 0).mean()
    print(f"Epoch {epoch + 1}/{n_epochs} - Avg Reward: {avg_reward:.2f} - Loss: {loss.item():.4f}")

    # # Learning rate decay (optional)
    # for param_group in optimizer.param_groups:
    #     param_group['lr'] = learning_rate * (0.99 ** epoch)



Epoch 1/1000 - Avg Reward: 5.68 - Loss: -0.1488
Epoch 2/1000 - Avg Reward: 5.96 - Loss: -0.3241
Epoch 3/1000 - Avg Reward: 7.50 - Loss: -0.3090
Epoch 4/1000 - Avg Reward: 7.48 - Loss: -0.2563
Epoch 5/1000 - Avg Reward: 16.94 - Loss: -0.3620
Epoch 6/1000 - Avg Reward: 15.91 - Loss: -0.3397
Epoch 7/1000 - Avg Reward: 13.76 - Loss: -0.3479
Epoch 8/1000 - Avg Reward: 14.18 - Loss: -0.3774
Epoch 9/1000 - Avg Reward: 15.48 - Loss: -0.3852
Epoch 10/1000 - Avg Reward: 15.98 - Loss: -0.3618
Epoch 11/1000 - Avg Reward: 15.85 - Loss: -0.3366
Epoch 12/1000 - Avg Reward: 14.91 - Loss: -0.3349
Epoch 13/1000 - Avg Reward: 12.41 - Loss: -0.3494
Epoch 14/1000 - Avg Reward: 8.67 - Loss: -0.3510
Epoch 15/1000 - Avg Reward: 6.05 - Loss: -0.3377
Epoch 16/1000 - Avg Reward: 4.74 - Loss: -0.3174
Epoch 17/1000 - Avg Reward: 4.24 - Loss: -0.3061
Epoch 18/1000 - Avg Reward: 4.53 - Loss: -0.3156
Epoch 19/1000 - Avg Reward: 5.19 - Loss: -0.3373
Epoch 20/1000 - Avg Reward: 6.67 - Loss: -0.3548
Epoch 21/1000 - Avg 

KeyboardInterrupt: 